# Python for GenAI- The 8 Skills You Actually Need

**Module 0 · Lesson 0.1**  
*Netsetos GenAI Engineering Course v2.0*

**In this lesson you will:**
- Async / Await - 5x Faster API Calls
- Generators & Yield - Streaming LLM Tokens
- Type Hints & Pydantic - Taming LLM Outputs
- Decorators - Auto-Retry, Cache, Rate Limit
- Env Variables & JSON - Secure Keys, Clean Data
- Error Handling & Resilience - LLM APIs Will Fail

---

> This notebook is generated from the published lesson HTML. The HTML is the source of truth - do not hand-edit this notebook. Re-run the generator if the lesson updates.


In [ ]:
# Reproducibility - the lesson uses seeded random values throughout

> 🔧 **Analogy**
>
> **You don't need to know ALL of Python to build with GenAI.** Just like a carpenter doesn't need every tool in the hardware store - they need a drill, a saw, a level, and sandpaper. These are your 8 power tools for GenAI engineering. Each one solves a specific problem you'll face in Modules 3-14.
> **Where this analogy breaks down:** a carpenter's tools are physical and stable for decades. Python LLM tooling changes quarterly - Pydantic v2 broke v1 patterns in 2023, Anthropic SDK changed retry behavior in late 2025. The skills are durable; the libraries are not. Pin your versions.

> 📦 **Info**
>
> 🎮 The production pattern
> The examples in this lesson are built around a small **LLM client** - the thin wrapper every GenAI app puts in front of a model. Here it's a stub: hardcoded responses, no real model calls. Each of the skills below maps to a piece a real production client needs: async for parallel message handling, Pydantic for structured tool output, decorators for retry/cache/observability. Later modules (5-14) reuse these exact patterns, but each lesson stands on its own - you're learning durable skills, not carrying one app forward across the whole course.

### Async / Await - 5x Faster API Calls

LLM calls take 2-5 seconds each. Sequential = slow. Async = parallel.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

**Setup.** your app needs to summarize 5 user messages at once - the user just opened a long thread and you want the summaries to render before they finish scrolling. Sequential calls take 5 × 2 seconds = 10 seconds. The user gives up at 4.

**Decision point.** Threading? The Python GIL plus the I/O-bound nature of LLM calls means threads add overhead without much speedup. Multiprocessing? Heavy for I/O work and forks the API client. Async is the right tool: one event loop, one thread, 5 awaitable coroutines running concurrently.

> ☕ **Analogy**
>
> **Ordering chai for 5 people.** Sequential: order one, wait 3 minutes, get it, order next. Total: 15 minutes. Async: order all 5 at once, wait 3 minutes, get all 5. Total: 3 minutes. **Async doesn't make each chai faster - it makes ALL chai happen at the same time.**
> **Where this analogy breaks down:** if you order 100 chais and the stall only has one stove, you're back to sequential - bottlenecked by the shared resource. Same with LLM calls behind a single connection pool or a single rate-limit bucket. Async parallelism is only as wide as your bottleneck allows.

### What you're seeing

5 LLM calls run one after the other. Each takes 2 seconds. Total: 10 seconds. The user gives up at 4. This is what async fixes.

**📝 `async_demo.py`** - *asyncio*

In [ ]:
# =============================================
# parallel message summarization
# Sequential: 10 seconds. Async: 2 seconds.
# =============================================

import asyncio, time

# Simulate an LLM API call (takes 2 seconds)
async def summarize_message(message: str, msg_id: int) -> str:
    print(f"  Msg {msg_id}: summarizing '{message[:30]}...'")
    await asyncio.sleep(2)  # simulated network wait
    return f"Summary of msg {msg_id}"

# ## Sequential (slow) ##
async def sequential():
    start = time.time()
    messages = ["What's a transformer?", "Explain RAG", "What is fine-tuning?", "Define embeddings", "What's an agent?"]
    results = []
    for i, m in enumerate(messages):
        r = await summarize_message(m, i+1)
        results.append(r)
    print(f"  Sequential: {time.time()-start:.1f}s for {len(results)} calls\n")

# ## Async (fast) ##
async def parallel():
    start = time.time()
    messages = ["What's a transformer?", "Explain RAG", "What is fine-tuning?", "Define embeddings", "What's an agent?"]
    tasks = [summarize_message(m, i+1) for i, m in enumerate(messages)]
    results = await asyncio.gather(*tasks)  # magic line
    print(f"  Async:      {time.time()-start:.1f}s for {len(results)} calls")

print("Sequential:")
asyncio.run(sequential())
print("Parallel:")
asyncio.run(parallel())

# Output:
# Sequential:
#   Msg 1: summarizing 'What's a transformer?...'
#   Msg 2: summarizing 'Explain RAG...'
#   ... (5 lines printed one every 2s)
#   Sequential: 10.2s for 5 calls
# Parallel:
#   Msg 1..5: (all printed in <0.01s)
#   Async:      2.1s for 5 calls   # ~4.9x speedup

#### 💡 What just happened

⚡ What just happened?
  Sequential: 5 calls × 2 seconds = **10 seconds**. Async: all 5 calls fire at once, wait for the slowest (2 seconds) = **2 seconds**. The magic line is `asyncio.gather(*tasks)` - it runs all coroutines concurrently. **Forward hook:** Module 7 (LLM API Engineering) productionizes this pattern with `httpx.AsyncClient`, timeouts, and multi-provider routing. Lesson 13.4 (Latency Optimization) measures the ceiling on async speedup.

### Generators & Yield - Streaming LLM Tokens

LLMs generate one token at a time. Generators let you process each token as it arrives.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

### What you're seeing

A normal function builds the entire output, then returns. The user waits 1.8 seconds with a blank screen, then everything appears at once. Total time: 1.8s. Perceived latency: 1.8s.

**📝 `streaming_demo.py`** - *Python*

In [ ]:
# =============================================
# stream tokens to the user as they arrive
# yield sends one item at a time instead of
# waiting to build the entire response.
# =============================================

import time

# ## Normal function: waits until EVERYTHING is ready ##
def demo_reply_normal() -> str:
    words = ["The", "future", "of", "AI", "is", "bright"]
    result = []
    for word in words:
        time.sleep(0.3)  # simulate generation time
        result.append(word)
    return " ".join(result)  # user waits 1.8s, then sees everything

# ## Generator: streams word by word ##
def reply_stream():
    words = ["The", "future", "of", "AI", "is", "bright"]
    for word in words:
        time.sleep(0.3)
        yield word  # sends each word IMMEDIATELY

# Normal: user sees nothing for 1.8s, then everything
print("Normal function:")
start = time.time()
result = demo_reply_normal()
print(f"  {result} (waited {time.time()-start:.1f}s)\n")

# Generator: user sees words appear one by one
print("Generator (streaming):")
start = time.time()
print("  ", end="")
for word in reply_stream():
    print(word, end=" ", flush=True)  # appears immediately
print(f"\n  (first word at 0.3s, total {time.time()-start:.1f}s)")

# Output:
# Normal function:
#   The future of AI is bright (waited 1.8s)
# Generator (streaming):
#   The future of AI is bright
#   (first word at 0.3s, total 1.9s)

#### 💡 What just happened

⚡ What just happened?
  Normal function: user stares at a blank screen for 1.8 seconds, then everything appears at once. Generator with `yield`: user sees the first word in 0.3 seconds, then each word appears progressively. **Same total time - but the perceived latency drops from 1.8s to 0.3s.** This is exactly how ChatGPT's streaming works. Python's async generator spec ([PEP 525](https://peps.python.org/pep-0525/)) extends this to async sources - the foundation for streaming from real LLM APIs in Module 7.

### Type Hints & Pydantic - Taming LLM Outputs

LLMs return messy text. Pydantic forces it into clean, validated Python objects.

**Setup.** the client asks the LLM to classify each user message into `{question, command, smalltalk, complaint, other}` with a confidence score. The model returns JSON. Sometimes it returns `"confidence": 95` (intended as percentage; schema wants 0-1). Sometimes it returns `"category": "QUESTION"` (uppercase; downstream string-match breaks).

**Decision point.** Use `json.loads()` (validates syntax only) or Pydantic's `BaseModel.model_validate_json()` (validates types AND constraints)? Production teams have shipped pipelines that silently accepted bad LLM output and skewed metrics for weeks. The schema is what catches this on the first call.

**📝 `validation_demo.py Pydantic`** - *v2*

In [ ]:
# =============================================
# message classifier - validate LLM JSON
# Pydantic v2 (pip install pydantic>=2.13)
# =============================================

from pydantic import BaseModel, Field
from typing import Literal, Optional
import json

# Define WHAT the LLM output should look like
class MessageClassification(BaseModel):
    category: Literal["question", "command", "smalltalk", "complaint", "other"]
    confidence: float = Field(ge=0, le=1, description="Confidence 0-1")
    rationale: str = Field(max_length=200)
    needs_followup: bool

# Simulate LLM returning JSON text
llm_output = '{"category": "question", "confidence": 0.92, "rationale": "user asks about a topic", "needs_followup": true}'

# Parse and validate - Pydantic catches errors automatically
result = MessageClassification.model_validate_json(llm_output)
print(f"Category:    {result.category}")
print(f"Confidence:  {result.confidence:.0%}")
print(f"Needs reply: {result.needs_followup}")
print(f"Type check:  confidence is {type(result.confidence).__name__}")

# What if the LLM returns garbage?
print("\nBad LLM output:")
try:
    bad = MessageClassification.model_validate_json(
        '{"category": "QUESTION", "confidence": 95, "rationale": "x", "needs_followup": "maybe"}'
    )
except Exception as e:
    print(f"  Pydantic caught it:")
    print(f"  - 'QUESTION' not in Literal categories (must be lowercase)")
    print(f"  - confidence 95 is > le=1")
    print(f"  - 'maybe' is not a valid bool")

# Output:
# Category:    question
# Confidence:  92%
# Needs reply: True
# Type check:  confidence is float
# 
# Bad LLM output:
#   Pydantic caught it:
#   - 'QUESTION' not in Literal categories (must be lowercase)
#   - confidence 95 is > le=1
#   - 'maybe' is not a valid bool

#### 💡 What just happened

⚡ What just happened?
  Pydantic v2's `model_validate_json` ran the JSON through a Rust-backed validator (per [Pydantic v2 docs](https://docs.pydantic.dev/latest/)). The schema specified `Literal[...]` categories, a confidence range of 0-1, and explicit bool. When the LLM returned bad data, Pydantic reported every violation in one error - you fix all three in one round-trip to the model, not three round-trips. **Forward hook:** Module 5.5 (Structured Output & Tool Use) builds prompts that explicitly constrain LLM output to a Pydantic schema. Module 7.4 (AI Chat with FastAPI) uses the same models for request/response validation.

### Decorators - Auto-Retry, Cache, Rate Limit

LLM APIs fail. Decorators add retry logic, caching, and logging without changing your code.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

### What you're seeing

A single @retry wrapping llm_call. The call enters retry's wrapper, which calls the inner function. If it fails, retry catches, waits, and tries again. After max_attempts failures, the exception propagates out.

#### 💡 What just happened

⚡ What just happened?
`@retry` automatically retries the function up to 3 times if it fails. `@timer` measures execution time. The function itself doesn't know about retrying or timing - the decorators add those superpowers externally. **Forward hook:** Module 12 (Production Deployment) builds a full AI gateway using stacked decorators for retry, cache, rate-limit, observability, and audit logging.

### Env Variables & JSON - Secure Keys, Clean Data

Never hardcode API keys. Always validate JSON from LLMs.

**📝 `demo_env.py`** - *Essential*

In [ ]:
# =============================================
# secret management + JSON parsing
# Never hardcode API keys
# =============================================

import os, json

# ## Environment variables ##
# Set in terminal: export ANTHROPIC_API_KEY="sk-ant-..."
# Or in .env file with python-dotenv (pip install python-dotenv)

api_key = os.getenv("ANTHROPIC_API_KEY", "NOT_SET")
print(f"API key: {api_key[:8]}... (first 8 chars only)")

if api_key == "NOT_SET":
    print("Set your key: export ANTHROPIC_API_KEY='sk-ant-...'")

# ## Safely parse JSON from LLM output ##
# LLMs sometimes wrap JSON in ```json ... ``` fences
def safe_parse_json(text: str) -> dict:
    """Parse JSON from LLM output, handling markdown fences."""
    cleaned = text.strip()
    if cleaned.startswith("```json"):
        cleaned = cleaned[7:]
    if cleaned.startswith("```"):
        cleaned = cleaned[3:]
    if cleaned.endswith("```"):
        cleaned = cleaned[:-3]
    try:
        return json.loads(cleaned.strip())
    except json.JSONDecodeError as e:
        print(f"Invalid JSON: {e}")
        return {}

# Test with messy LLM output
messy_output = '```json\n{"intent": "question", "score": 0.92}\n```'
parsed = safe_parse_json(messy_output)
print(f"\nParsed: {parsed}")
print(f"Intent: {parsed.get('intent')}")
print(f"Score:  {parsed.get('score')}")

# Output:
# API key: sk-ant-a... (first 8 chars only)
# 
# Parsed: {'intent': 'question', 'score': 0.92}
# Intent: question
# Score:  0.92

### Error Handling & Resilience - LLM APIs Will Fail

Rate limits, timeouts, server errors. Your app must handle all of them gracefully.

> 🚆 **Analogy**
>
> **LLM APIs are like a busy transit network.** Usually they work. Sometimes they're delayed (timeout). Sometimes they're at capacity and refuse more passengers (429 rate limit). Occasionally the whole station shuts down (5xx server error). A good traveler has backup plans for all of these. Your code needs the same resilience.
> **Where this analogy breaks down:** the transit metaphor implies a fixed schedule. Real APIs are stochastic - 429s can spike with adversarial bursts, 5xxs cluster during provider incidents, and degraded modes (slow but not failing) are common. Your retry math needs jitter, your timeouts need to be conservative, and your monitoring needs to distinguish "API slower" from "API failing" before either becomes a problem.

The production pattern: catch specific HTTP errors with different policies, implement **exponential backoff with jitter**, and provide a **model fallback chain** (try Claude Sonnet -> fall back to Claude Haiku -> fall back to cached response). Per the [Anthropic API rate limits guide (2026)](https://www.respan.ai/articles/anthropic-api-rate-limits), the SDK auto-retries 408/409/429/5xx a default of `max_retries=2` times with exponential backoff, and when the API returns a `Retry-After` (or `retry-after-ms`) header it uses that value to time the wait. When you need full control over retry timing - custom backoff, jitter, or a shared retry budget - set `max_retries=0` on the client and implement retry yourself.

**📝 `resilience_demo.py`** - *Production*

In [ ]:
# =============================================
# LLM error handling
# Pattern: structured errors + backoff + fallback chain
# =============================================

import time, random

# ## 1. Structured exception handling for LLM APIs ##
# Different errors need different responses:
#   429 Rate Limit  -> wait and retry (exponential backoff)
#   500 Server Error -> retry (might be temporary)
#   401 Auth Error  -> FAIL FAST (don't retry, key is wrong)
#   Timeout         -> retry with longer timeout

class RateLimitError(Exception): pass
class ServerError(Exception): pass
class AuthError(Exception): pass

def call_llm_unreliable(prompt: str) -> str:
    """Simulates an unreliable LLM API."""
    roll = random.random()
    if roll < 0.3: raise RateLimitError("429: Too Many Requests")
    if roll < 0.4: raise ServerError("500: Internal Server Error")
    if roll < 0.42: raise AuthError("401: Invalid API Key")
    time.sleep(0.5)
    return f"Response for: {prompt}"

# ## 2. Exponential backoff with jitter ##
def retry_with_backoff(func, prompt, max_retries=3):
    """Retry with exponential backoff. Fail fast on auth errors."""
    for attempt in range(max_retries):
        try:
            return func(prompt)
        except AuthError:
            print(f"  Auth error - FAIL FAST (don't retry)")
            raise  # Never retry auth errors
        except RateLimitError:
            wait = (2 ** attempt) + random.uniform(0, 1)  # jitter prevents thundering herd
            print(f"  Rate limited. Retry {attempt+1}/{max_retries} in {wait:.1f}s")
            time.sleep(wait)
        except ServerError:
            print(f"  Server error. Retry {attempt+1}/{max_retries}")
            time.sleep(1)
    raise Exception(f"Failed after {max_retries} retries")

# ## 3. Model fallback chain ##
def call_with_fallback(prompt: str) -> str:
    """Try primary, fall back to cheaper, then to cache."""
    models = ["claude-sonnet-4-6", "claude-haiku-4-6", "cached"]
    for model in models:
        try:
            print(f"  Trying {model}...")
            if model == "cached":
                return "[Cached fallback response]"
            return retry_with_backoff(call_llm_unreliable, prompt)
        except Exception as e:
            print(f"  {model} failed: {e}")
    return "Service temporarily unavailable"

# Test
print("Calling LLM with full error handling:\n")
result = call_with_fallback("What is async/await?")
print(f"\nFinal result: {result}")
print(f"\nIn production: use 'tenacity' library instead of hand-rolled retry")
print(f"  pip install tenacity")
print(f"  @retry(wait=wait_exponential(min=1, max=60), stop=stop_after_attempt(3))")

# Output:
# Calling LLM with full error handling:
#   Trying claude-sonnet-4-6...
#   Rate limited. Retry 1/3 in 1.4s
#   Server error. Retry 2/3
#   ...
#   Final result: Response for: What is async/await?

#### 💡 What just happened

⚡ What just happened?
  Three resilience layers: **(1) Structured error handling** - different errors get different treatment. Auth errors fail fast. Rate limits wait and retry. **(2) Exponential backoff with jitter** - wait 1s, 2s, 4s... plus random jitter so 1000 clients don't all retry at the same second (the "thundering herd" pattern). **(3) Model fallback chain** - if Sonnet is down, try Haiku, then serve a cached response. This is the architecture every production LLM app uses. Real-world pattern: during a provider outage or partial degradation, apps with fallback chains stay up by routing non-critical paths to a smaller model (for example Haiku) or a cached response.

### The Production LLM Call - All Skills Combined

Every skill in this lesson combined into one production-ready pattern.

Here's the architecture of every LLM call in Modules 5-14. It combines async, Pydantic, retry, caching, streaming, env vars, and error handling into one clean pipeline. It's the shape of the LLM-call code you'll see reused across those modules.

**📝 `production_demo.py`** - *Capstone*

In [ ]:
# =============================================
# production LLM call - all 7 skills combined
# This is exactly how Modules 5-14 make LLM calls
# =============================================

import asyncio, os, json, time, functools
from pydantic import BaseModel, Field

# ## Skill 5: Env vars ##
API_KEY = os.getenv("ANTHROPIC_API_KEY", "demo-key")
if API_KEY == "demo-key":  # no real key set
    print("No ANTHROPIC_API_KEY set - running in demo mode (simulated calls).\n")

# ## Skill 3+4: Pydantic validates LLM output ##
class LLMResponse(BaseModel):
    answer: str = Field(..., min_length=1)
    confidence: float = Field(..., ge=0, le=1)
    tokens: int = Field(..., ge=1)

# ## Skill 4: Decorators stack (cache wraps async) ##
def cache(func):
    _c = {}
    @functools.wraps(func)
    async def wrapper(*a):
        k = str(a)
        if k in _c: return _c[k]
        r = await func(*a); _c[k] = r; return r
    return wrapper

# ## Skill 1+6: Async + error handling + retry ##
@cache
async def llm_call(prompt: str) -> LLMResponse:
    for attempt in range(3):
        try:
            await asyncio.sleep(0.5)  # simulated API call
            raw = {"answer": f"GenAI uses transformers for {prompt}",
                   "confidence": 0.92, "tokens": 45}
            return LLMResponse(**raw)  # Pydantic validates
        except Exception:
            if attempt < 2: await asyncio.sleep(2 ** attempt)
            else: raise

# ## Skill 2: Generator for streaming ##
def stream(text, delay=0.05):  # simulated 50ms per word
    for word in text.split():
        time.sleep(delay)
        yield word + " "

# ## Run the pipeline ##
async def main():
    prompts = ["RAG", "agents", "MCP"]

    # Parallel calls with async
    start = time.time()
    results = await asyncio.gather(*[llm_call(p) for p in prompts])
    print(f"3 calls in {time.time()-start:.1f}s (parallel)\n")

    # Repeat "RAG" - the first call already populated the cache,
    # so this returns instantly. (A plain dict cache only dedupes
    # AFTER a call finishes - concurrent duplicates in one gather()
    # still both run; real coalescing needs a lock or shared future.)
    start = time.time()
    cached = await llm_call("RAG")
    print(f"Repeat 'RAG' in {time.time()-start:.3f}s (from cache)\n")
    results.append(cached)

    # Stream each result
    for r in results:
        print(f"[{r.confidence:.0%} conf, {r.tokens} tokens] ", end="")
        for w in stream(r.answer): print(w, end="", flush=True)
        print()

await main()

# Output:
# No ANTHROPIC_API_KEY set - running in demo mode (simulated calls).
# 
# 3 calls in 0.5s (parallel)
# 
# Repeat 'RAG' in 0.000s (from cache)
# 
# [92% conf, 45 tokens] GenAI uses transformers for RAG
# [92% conf, 45 tokens] GenAI uses transformers for agents
# [92% conf, 45 tokens] GenAI uses transformers for MCP
# [92% conf, 45 tokens] GenAI uses transformers for RAG    # served from cache

#### 💡 What just happened

⚡ What just happened?
  Every skill combined: **env vars** load the API key. **Pydantic** validates the response. **Decorators** add caching. **Async** runs 3 calls in parallel; the repeat "RAG" then returns from cache. **Error handling** retries on failure. **Generators** stream each word. This is the exact architecture you'll build in Module 7.4 (AI Chat with FastAPI) and refine through Module 12.3 (AI Gateway Pattern).

## Quick Reference - When You'll Use Each Skill

**📦 **Async/Await****

| Python Skill | Where You'll Use It | Importance |
|---|---|---|
| Async/Await | Module 7 (multi-provider), Module 13 (parallel batch), Module 12 (scaling) | Critical - most LLM work is I/O |
| Generators | Module 7 (streaming), Module 12 (SSE), every chat application | Critical - all streaming UIs |
| Pydantic | Module 5 (validated JSON), Module 7 (FastAPI), Module 10 (MCP/tool schemas) | Critical - industry standard |
| Decorators | Module 12 (retry, cache, rate limit), Module 7 (observability), Module 10 (MCP tools) | Critical - production patterns |
| Error Handling | Module 7 (API resilience), Module 12 (circuit breakers), Module 13 (fallback chains) | Critical - apps WILL crash without it |
| Type Hints | Module 5 (structured output), Module 10 (MCP schemas), Module 7 (FastAPI) | High |
| Env Variables | EVERY module - API keys are never hardcoded | High - security baseline |
| JSON Parsing | Module 5 (LLM output), Module 8 (RAG data), Module 10 (MCP messages) | High |

> 💡 **Info**
>
> 💡 Pro tip - How to use the exercises below
>   The exercises below are ranked Easy -> Hard. Try solving them in your **Colab notebook** or local Python first. If you get stuck, check the `.py` solution files. Don't just read the solutions - type the code yourself. Muscle memory matters for interviews.

*Practice exercises are in the companion practice notebook.*

> ℹ️ **Info**
>
> 📚 Continue with
> -> **Lesson 1.1 - Math Foundations: The Basics**: vectors, cosine similarity, softmax, temperature. Understanding the math is how you debug LLM behavior.  
>   -> **Module 5 - Prompt Engineering**: deepens Pydantic for structured output (5.5) and adds DSPy for prompt optimization (5.6).  
>   -> **Module 7 - LLM API Engineering**: productionizes this retry + fallback pattern in FastAPI with real Anthropic/OpenAI clients.

---

## 🎓 Summary

You've completed the practical part of **Python for GenAI- The 8 Skills You Actually Need**.

### Next steps:
1. Re-run cells with different parameters to build intuition
2. Try the practice exercises (see `Lesson_0.1_Practice.ipynb` if available)
3. Review the full HTML lesson for the complete narrative

*Generated from `lesson-0.1.html` - regenerate this notebook whenever the source HTML is updated.*
